# S27 — Evaluation and Safety

**Week 14 · Module 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s27_evaluation_and_safety.ipynb)

Every cell below is a worked example from the [S27 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s27/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s27.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s27.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Measure the noise before comparing models


*Expected output starts with:* `hidden=16  accs: ['0.863', '0.862', '0.860', '0.860', '0.864']`


In [ ]:
import torch
import torch.nn as nn

# Fixed dataset (its own seed, independent of training seeds):
# two overlapping Gaussian blobs in 2D
g = torch.Generator().manual_seed(0)
def blobs(n, gen):
    y = (torch.rand(n, generator=gen) < 0.5).long()
    centers = torch.tensor([[-0.8, -0.8], [0.8, 0.8]])
    x = centers[y] + torch.randn(n, 2, generator=gen)
    return x, y

x_tr, y_tr = blobs(2000, g)
x_te, y_te = blobs(1000, g)

def run(hidden, seed, steps=500):
    torch.manual_seed(seed)   # controls init AND minibatch order
    net = nn.Sequential(nn.Linear(2, hidden), nn.ReLU(), nn.Linear(hidden, 2))
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for step in range(steps):
        idx = torch.randint(0, len(x_tr), (64,))
        loss = nn.functional.cross_entropy(net(x_tr[idx]), y_tr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (net(x_te).argmax(1) == y_te).float().mean().item()

for hidden in [16, 32]:
    accs = torch.tensor([run(hidden, seed) for seed in range(5)])
    print(f"hidden={hidden:2d}  accs: {[f'{a:.3f}' for a in accs.tolist()]}")
    print(f"hidden={hidden:2d}  mean {accs.mean().item():.4f} +/- {accs.std().item():.4f} (std over 5 seeds)")

## Error bars from the test set: the bootstrap


*Expected output starts with:* `test n=1000: acc 0.8630  95% bootstrap CI [0.8410, 0.8840]  width 0.0430`


In [ ]:
import torch
import torch.nn as nn

# Same data and model as the seed-variance experiment; one trained model
g = torch.Generator().manual_seed(0)
def blobs(n, gen):
    y = (torch.rand(n, generator=gen) < 0.5).long()
    centers = torch.tensor([[-0.8, -0.8], [0.8, 0.8]])
    x = centers[y] + torch.randn(n, 2, generator=gen)
    return x, y

x_tr, y_tr = blobs(2000, g)
x_te, y_te = blobs(1000, g)

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
opt = torch.optim.Adam(net.parameters(), lr=1e-2)
for step in range(500):
    idx = torch.randint(0, len(x_tr), (64,))
    loss = nn.functional.cross_entropy(net(x_tr[idx]), y_tr[idx])
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    correct = (net(x_te).argmax(1) == y_te).float()   # 0/1 per test example

def bootstrap_ci(correct, n_boot=10000, gen=None):
    n = len(correct)
    idx = torch.randint(0, n, (n_boot, n), generator=gen)
    accs = correct[idx].mean(dim=1)                   # one accuracy per resample
    lo = torch.quantile(accs, 0.025).item()
    hi = torch.quantile(accs, 0.975).item()
    return correct.mean().item(), lo, hi

gb = torch.Generator().manual_seed(1)
acc, lo, hi = bootstrap_ci(correct, gen=gb)
print(f"test n=1000: acc {acc:.4f}  95% bootstrap CI [{lo:.4f}, {hi:.4f}]  "
      f"width {hi-lo:.4f}")
acc, lo, hi = bootstrap_ci(correct[:200], gen=gb)
print(f"test n= 200: acc {acc:.4f}  95% bootstrap CI [{lo:.4f}, {hi:.4f}]  "
      f"width {hi-lo:.4f}")

## How big a difference can your test set even see?


*Expected output starts with:* `power to detect a true accuracy gain (alpha = 0.05, one-sided)`


In [ ]:
import torch

torch.manual_seed(0)

def detect_rate(p_a, p_b, n, trials=3000, z_crit=1.645):
    """Fraction of experiments where a two-proportion z-test finds B > A
    significant at alpha = 0.05, with n test examples per model."""
    acc_a = torch.distributions.Binomial(n, torch.tensor(p_a)).sample((trials,)) / n
    acc_b = torch.distributions.Binomial(n, torch.tensor(p_b)).sample((trials,)) / n
    pool = (acc_a + acc_b) / 2
    se = (2 * pool * (1 - pool) / n).sqrt()
    z = (acc_b - acc_a) / se
    return (z > z_crit).float().mean().item()

print("power to detect a true accuracy gain (alpha = 0.05, one-sided)")
print(f"{'n test':>8} {'+0.5 pt':>8} {'+1 pt':>7} {'+2 pt':>7} {'+5 pt':>7} "
      f"{'false pos (+0)':>15}")
for n in [200, 500, 1000, 5000, 20000]:
    row = [detect_rate(0.85, 0.85 + d, n) for d in [0.005, 0.01, 0.02, 0.05]]
    fp = detect_rate(0.85, 0.85, n)
    print(f"{n:>8} {row[0]:>8.3f} {row[1]:>7.3f} {row[2]:>7.3f} {row[3]:>7.3f} "
          f"{fp:>15.3f}")

## Comparing many things: the multiple-comparisons trap


*Expected output starts with:* `20 identical configs, different seeds:`


In [ ]:
import torch
import torch.nn as nn

# 20 runs of the SAME config (hidden=16), differing only in seed
g = torch.Generator().manual_seed(0)
def blobs(n, gen):
    y = (torch.rand(n, generator=gen) < 0.5).long()
    centers = torch.tensor([[-0.8, -0.8], [0.8, 0.8]])
    x = centers[y] + torch.randn(n, 2, generator=gen)
    return x, y

x_tr, y_tr = blobs(2000, g)
x_te, y_te = blobs(1000, g)

def run(seed):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for step in range(500):
        idx = torch.randint(0, len(x_tr), (64,))
        loss = nn.functional.cross_entropy(net(x_tr[idx]), y_tr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (net(x_te).argmax(1) == y_te).float().mean().item()

accs = torch.tensor([run(seed) for seed in range(20)])
best, worst = accs.max().item(), accs.min().item()
print(f"20 identical configs, different seeds:")
print(f"  mean {accs.mean().item():.4f}  std {accs.std().item():.4f}")
print(f"  best seed  {accs.argmax().item():2d}: {best:.4f}")
print(f"  worst seed {accs.argmin().item():2d}: {worst:.4f}")
print(f"  best - worst = {best - worst:.4f}  "
      f"({(best - worst) * 1000:.0f} test examples out of 1000)")

## Making the correction: Bonferroni and friends


*Expected output starts with:* `k identical configs vs baseline, n = 1000 test examples, alpha = 0.05`


In [ ]:
import torch

torch.manual_seed(0)

# A hyperparameter sweep where NOTHING works: k candidate configs, every one
# with the same true accuracy as the baseline (0.86). Any "significant win"
# is a false positive by construction.
def sweep_experiment(k, n=1000, trials=2000, alpha=0.05):
    p = 0.86
    base = torch.distributions.Binomial(n, torch.tensor(p)).sample((trials, 1)) / n
    cand = torch.distributions.Binomial(n, torch.tensor(p)).sample((trials, k)) / n
    pool = (base + cand) / 2
    se = (2 * pool * (1 - pool) / n).sqrt()
    z = (cand - base) / se
    z_raw = torch.tensor(1.645)                       # one-sided alpha = 0.05
    # Bonferroni: test each of the k comparisons at alpha / k
    z_bonf = torch.distributions.Normal(0., 1.).icdf(torch.tensor(1 - alpha / k))
    fwer_raw = (z > z_raw).any(dim=1).float().mean().item()
    fwer_bonf = (z > z_bonf).any(dim=1).float().mean().item()
    n_sig = (z > z_raw).float().sum(dim=1).mean().item()
    return fwer_raw, fwer_bonf, n_sig

print("k identical configs vs baseline, n = 1000 test examples, alpha = 0.05")
print(f"{'k':>4} {'P(any raw win)':>15} {'avg raw wins':>13} {'P(any Bonf win)':>16}")
for k in [1, 5, 20, 100]:
    raw, bonf, avg = sweep_experiment(k)
    print(f"{k:>4} {raw:>15.3f} {avg:>13.2f} {bonf:>16.3f}")

## How bias enters: shortcuts in the loss landscape


*Expected output starts with:* `learned weights: skill 2.3325   attribute s 4.3128`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def make_data(n, p_match):
    """Label depends on a real-valued 'skill' signal. A binary attribute s
    matches the label with probability p_match (spurious correlation)."""
    skill = torch.randn(n)
    y = (skill + 0.7 * torch.randn(n) > 0).float()      # noisy function of skill
    match = (torch.rand(n) < p_match).float()
    s = match * y + (1 - match) * (1 - y)               # s = y w.p. p_match
    x = torch.stack([skill, s], dim=1)
    return x, y

x_tr, y_tr = make_data(4000, p_match=0.9)   # strong correlation in training
x_te, y_te = make_data(4000, p_match=0.5)   # correlation absent at deployment

model = nn.Linear(2, 1)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(2000):
    loss = nn.functional.binary_cross_entropy_with_logits(model(x_tr).squeeze(1), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()

w = model.weight.detach().squeeze()
print(f"learned weights: skill {w[0].item():.4f}   attribute s {w[1].item():.4f}")

def acc(x, y):
    with torch.no_grad():
        return ((model(x).squeeze(1) > 0).float() == y).float().mean().item()

print(f"train accuracy (correlation 0.9): {acc(x_tr, y_tr):.4f}")
print(f"test accuracy (correlation 0.5):  {acc(x_te, y_te):.4f}")
agree = x_te[:, 1] == y_te
print(f"  test acc when s agrees with y:    {acc(x_te[agree], y_te[agree]):.4f}")
print(f"  test acc when s disagrees with y: {acc(x_te[~agree], y_te[~agree]):.4f}")

## Two mitigations, measured


*Expected output starts with:* `baseline (skill + s)    test acc 0.6725   s==y 0.9879   s!=y 0.3627   gap +0.6252`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def make_data(n, p_match):
    skill = torch.randn(n)
    y = (skill + 0.7 * torch.randn(n) > 0).float()
    match = (torch.rand(n) < p_match).float()
    s = match * y + (1 - match) * (1 - y)
    x = torch.stack([skill, s], dim=1)
    return x, y

x_tr, y_tr = make_data(4000, p_match=0.9)
x_te, y_te = make_data(4000, p_match=0.5)

def train_eval(name, features, weights=None):
    torch.manual_seed(0)
    model = nn.Linear(features.shape[1], 1)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for step in range(2000):
        logits = model(features).squeeze(1)
        loss = nn.functional.binary_cross_entropy_with_logits(
            logits, y_tr, weight=weights)
        opt.zero_grad(); loss.backward(); opt.step()
    x_eval = x_te if features.shape[1] == 2 else x_te[:, :1]
    with torch.no_grad():
        pred = (model(x_eval).squeeze(1) > 0).float()
    agree = x_te[:, 1] == y_te
    overall = (pred == y_te).float().mean().item()
    a = (pred[agree] == y_te[agree]).float().mean().item()
    d = (pred[~agree] == y_te[~agree]).float().mean().item()
    print(f"{name:22}  test acc {overall:.4f}   s==y {a:.4f}   s!=y {d:.4f}   "
          f"gap {a - d:+.4f}")

# Baseline: both features, unweighted (the shortcut learner)
train_eval("baseline (skill + s)", x_tr)

# Fix 1: drop the attribute entirely
train_eval("drop s", x_tr[:, :1])

# Fix 2: keep s, but reweight the four (y, s) cells so s carries no signal
wts = torch.zeros(len(y_tr))
for yv in [0.0, 1.0]:
    for sv in [0.0, 1.0]:
        cell = (y_tr == yv) & (x_tr[:, 1] == sv)
        wts[cell] = len(y_tr) / (4 * cell.sum())     # equalize cell mass
train_eval("reweight (y, s) cells", x_tr, weights=wts)

## Models remember: membership inference


*Expected output starts with:* `                 model  attack acc  train acc  test acc  loss in  loss out`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Two-moons classification with label noise, so memorization has something to grab.
def moons(n, gen):
    t = torch.rand(n, generator=gen) * torch.pi
    x = torch.where(
        (torch.arange(n) % 2 == 0).unsqueeze(1),
        torch.stack([torch.cos(t), torch.sin(t)], dim=1),
        torch.stack([1 - torch.cos(t), 0.5 - torch.sin(t)], dim=1),
    ) + 0.2 * torch.randn(n, 2, generator=gen)
    y = (torch.arange(n) % 2).long()
    flip = torch.rand(n, generator=gen) < 0.1          # 10% mislabeled
    return x, torch.where(flip, 1 - y, y)

g = torch.Generator().manual_seed(0)
x_tr, y_tr = moons(150, g)      # small training set: easy to memorize
x_out, y_out = moons(150, g)    # non-members from the same distribution

def train(steps, weight_decay):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                        nn.Linear(256, 2))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=weight_decay)
    for step in range(steps):
        loss = nn.functional.cross_entropy(net(x_tr), y_tr)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

def attack(net):
    """Loss-threshold membership inference: guess 'member' when the model's
    loss on the example is below a threshold (best threshold reported)."""
    with torch.no_grad():
        l_in = nn.functional.cross_entropy(net(x_tr), y_tr, reduction="none")
        l_out = nn.functional.cross_entropy(net(x_out), y_out, reduction="none")
    accs = []
    for thresh in torch.cat([l_in, l_out]).sort().values:
        hits = (l_in <= thresh).float().sum() + (l_out > thresh).float().sum()
        accs.append(hits / (len(l_in) + len(l_out)))
    acc = torch.stack(accs).max()
    with torch.no_grad():
        tr_acc = (net(x_tr).argmax(1) == y_tr).float().mean()
        te_acc = (net(x_out).argmax(1) == y_out).float().mean()
    return acc.item(), tr_acc.item(), te_acc.item(), l_in.mean().item(), l_out.mean().item()

print(f"{'model':>22} {'attack acc':>11} {'train acc':>10} {'test acc':>9} "
      f"{'loss in':>8} {'loss out':>9}")
for name, steps, wd in [("overfit (5k steps)", 5000, 0.0),
                        ("early stop (300)", 300, 0.0),
                        ("weight decay 1e-2", 5000, 1e-2)]:
    net = train(steps, wd)
    a, tr, te, li, lo = attack(net)
    print(f"{name:>22} {a:>11.3f} {tr:>10.3f} {te:>9.3f} {li:>8.4f} {lo:>9.4f}")
print("(attack acc 0.5 = coin flip; the attacker knows nothing)")

## Differential privacy: a knob with a price


*Expected output starts with:* `typical error added to a released fraction (median |noise|)`


In [ ]:
import torch

torch.manual_seed(0)

# The statistic to release: what fraction of a cohort has some sensitive attribute.
# Changing ONE person changes the fraction by at most 1/n -- the "sensitivity".
# The Laplace mechanism adds noise scaled to sensitivity / epsilon.
def released_error(n, epsilon, trials=20000):
    sensitivity = 1.0 / n
    scale = sensitivity / epsilon
    noise = torch.distributions.Laplace(0.0, scale).sample((trials,))
    return noise.abs().median().item()   # typical |error| added to the true fraction

print("typical error added to a released fraction (median |noise|)")
print(f"{'epsilon':>8} {'n=100':>9} {'n=1,000':>9} {'n=100,000':>10}")
for eps in [0.01, 0.1, 1.0, 10.0]:
    row = [released_error(n, eps) for n in [100, 1000, 100000]]
    print(f"{eps:>8.2f} {row[0]:>9.4f} {row[1]:>9.4f} {row[2]:>10.6f}")

# What epsilon means: likelihood ratio for any output is bounded by e^epsilon.
for eps in [0.1, 1.0, 10.0]:
    print(f"epsilon = {eps:>5.1f}: an adversary's odds about any one person "
          f"change by at most a factor of {torch.tensor(eps).exp().item():.2f}")

## Try it yourself

1. Extend the seed-variance experiment to 20 seeds per configuration and check whether the hidden=16 vs. hidden=32 ordering from 5 seeds holds. Does the gap grow, shrink, or change sign?
2. Implement a *paired* bootstrap: train both configurations once, keep both correctness vectors, resample indices once per iteration, and compute the CI of the *difference* in accuracy. Does the interval contain zero?
3. Simulate contamination: move 100 test examples into the training set (keeping them in the test set too), retrain, and measure the test accuracy change. This is the mechanism behind inflated benchmark scores, at laptop scale.
4. Using the 20-run experiment, compute how often the best of `k` identical configs beats the worst by more than 1 point, for `k` in 2, 5, 10, 20 (resample subsets from your 20 runs). At what `k` does a 1-point "improvement" become unremarkable?
5. Extend the power simulation to a *paired* design: draw per-item correctness for both models with a shared item-difficulty component (e.g., both models correct on the same "easy" 70% of items, differing only on the rest), and detect with McNemar-style disagreement counts. How much does pairing improve the power at n = 1000 for a +2-point effect?
6. In the Bonferroni simulation, give one of the `k` candidates a *real* +3-point improvement. Measure how often it is the config that gets crowned (a) uncorrected, (b) with Bonferroni. What fraction of the corrected sweeps still find it — that is, what did the correction cost you in power?


---

Full discussion of everything above: [S27 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s27/).
